# LexForge AI — Stage 1: QLoRA Fine-Tuning

**Domain-Adaptive Legal Intelligence Platform**

| Component | Choice |
|-----------|--------|
| Base model | `Qwen/Qwen2.5-3B-Instruct` |
| Dataset | `theatticusproject/cuad-qa` (CUAD v1 reformatted as QA) |
| Method | QLoRA (4-bit NF4) + LoRA adapters via PEFT |
| Trainer | TRL `SFTTrainer` with chat-template formatting |

**Hardware target:** single GPU with ≥ 16 GB VRAM (T4, A10, 3090, 4090, L4, A100).

**Expected time:** ~2-4 hours for 1 epoch on a T4, ~45 min on an A100.

---

## Design rationale

- **Dataset:** CUAD-QA is the same CUAD v1 you planned in your spec, reformatted as extractive QA pairs (~22k train, ~4k test) covering all 41 clause types. Right size to fine-tune on a single GPU in a few hours.
- **4-bit NF4 quantization:** cuts the 3B model from ~6 GB to ~2 GB VRAM. Double-quant shaves another ~0.4 GB.
- **LoRA rank 32, alpha 64, all-linear targeting:** rank 32 is the sweet spot for 3B models (7B would use 64). Targeting all seven linear projections (attention + MLP) consistently outperforms attention-only in the QLoRA paper.
- **`paged_adamw_8bit` + gradient checkpointing:** the two memory tricks that make QLoRA fit on consumer GPUs.
- **TRL `SFTTrainer` with chat templates:** ensures we train in the exact `<|im_start|>...<|im_end|>` format Qwen expects at inference.

## 1. Install dependencies

Run this once per environment. If you're on Colab, a factory-reset runtime is recommended afterwards so the new CUDA libraries are picked up.

In [ ]:
!pip install -q \
    "torch>=2.3.0" \
    "transformers>=4.45.0" \
    "accelerate>=0.34.0" \
    "datasets>=2.20.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.44.0" \
    "trl>=0.11.0" \
    "mlflow>=2.16.0" \
    sentencepiece protobuf tiktoken

# Flash-attention 2 — comment out the next line if your GPU is pre-Ampere (T4, V100)
# or if the wheel build fails. Training will still work without it, just slower.
!pip install -q flash-attn --no-build-isolation

In [ ]:
# Sanity check: verify GPU is visible
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name       : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bfloat16 ok    : {torch.cuda.is_bf16_supported()}")

## 2. Configuration

All hyperparameters live in one place so you can tune without hunting.

In [ ]:
import os
import torch

MODEL_ID      = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID    = "theatticusproject/cuad-qa"
OUTPUT_DIR    = "./lexforge-qwen2.5-3b-cuad"
HF_REPO_ID    = None   # e.g. "your-username/lexforge-qwen2.5-3b-cuad" to push to Hub

MAX_SEQ_LEN   = 2048
NUM_EPOCHS    = 1       # CUAD-QA has ~22k train examples; 1 epoch is a solid baseline
BATCH_SIZE    = 2       # per-device; bump to 4 on a 24GB+ card
GRAD_ACCUM    = 8       # effective batch = 2 * 8 = 16
LEARNING_RATE = 2e-4    # standard LoRA LR; higher than full-FT
WARMUP_RATIO  = 0.03
LOGGING_STEPS = 25
SAVE_STEPS    = 500
EVAL_STEPS    = 500
SEED          = 42

# Precision: bf16 on Ampere+ (A10/3090/4090/A100/L4). Flip to fp16 on T4/V100.
USE_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
USE_FP16 = not USE_BF16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# Flash-attention 2: requires Ampere+. Set to None (default eager/sdpa) otherwise.
ATTN_IMPL = "flash_attention_2" if USE_BF16 else "sdpa"

print(f"Compute dtype : {COMPUTE_DTYPE}")
print(f"Attention impl: {ATTN_IMPL}")

## 3. 4-bit quantization config (QLoRA)

This is the core of QLoRA — the base model is frozen at 4-bit NF4 precision while LoRA adapters train in bf16/fp16.

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                 # NF4 = Normal Float 4, QLoRA's key insight
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,            # quantize the quantization constants too
)

## 4. Tokenizer and base model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # right-pad for training; switch to left for generation

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocab size: {len(tokenizer)}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation=ATTN_IMPL,
)
model.config.use_cache = False         # required when gradient checkpointing is on
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 5. LoRA configuration

Targeting all seven linear layers (attention + MLP) as recommended by the QLoRA paper. Expect ~30M trainable params, about 1% of the total.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare the quantized model for k-bit training
# (casts layernorms to fp32, enables input grads, etc.)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,                       # rank — 32 is a good sweet spot for 3B models
    lora_alpha=64,              # scaling factor; alpha = 2*r is a common heuristic
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",       # attention
        "gate_proj", "up_proj", "down_proj",           # MLP
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~30M trainable / 3B total  ≈  1%

## 6. Dataset loading and formatting

CUAD-QA fields: `question`, `context` (contract excerpt), `answers` (list of spans).

We format each example as a chat conversation using Qwen's chat template, so the model learns to respond in exactly the format we'll prompt it with at inference.

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are LexForge AI, a legal-domain assistant specialized in contract analysis. "
    "Given a contract excerpt and a question about a specific clause or provision, "
    "answer precisely using only information from the excerpt. "
    "If the contract does not contain the information, respond exactly: "
    "'Not specified in the provided contract.'"
)

print("Loading CUAD-QA dataset...")
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)

In [ ]:
def format_example(example):
    """Turn a CUAD-QA row into a chat-templated string."""
    question = example["question"]
    context  = example["context"]
    answers  = example["answers"]["text"]

    # Ground-truth answer: first span, or "Not specified..." if empty
    answer = answers[0].strip() if answers and answers[0].strip() else \
             "Not specified in the provided contract."

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Contract excerpt:\n{context}\n\nQuestion: {question}"},
        {"role": "assistant", "content": answer},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Format train split
train_dataset = raw_dataset["train"].map(
    format_example,
    remove_columns=raw_dataset["train"].column_names,
    desc="Formatting train split",
)

# CUAD-QA has a 'test' split we use as held-out eval
eval_split_name = "test" if "test" in raw_dataset else "validation"
eval_dataset = raw_dataset[eval_split_name].map(
    format_example,
    remove_columns=raw_dataset[eval_split_name].column_names,
    desc="Formatting eval split",
)
# Use a 2k slice for eval to keep iteration fast
eval_dataset = eval_dataset.shuffle(seed=SEED).select(range(min(2000, len(eval_dataset))))

print(f"\nTrain examples: {len(train_dataset)}  |  Eval examples: {len(eval_dataset)}")
print("\n--- Sample formatted example ---")
print(train_dataset[0]["text"][:1200])

## 7. Training configuration

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",            # QLoRA's memory-efficient optimizer

    bf16=USE_BF16,
    fp16=USE_FP16,
    tf32=USE_BF16,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="mlflow",                   # matches your spec; "none" for local-only
    run_name="lexforge-qwen2.5-3b-cuad-qlora",

    # SFTConfig-specific
    max_seq_length=MAX_SEQ_LEN,
    packing=False,                        # packing=True is faster but can mix examples
    dataset_text_field="text",

    seed=SEED,
    dataloader_num_workers=4,
    remove_unused_columns=True,

    # Push final adapter to Hub if HF_REPO_ID is set
    push_to_hub=bool(HF_REPO_ID),
    hub_model_id=HF_REPO_ID,
    hub_strategy="end",
)

## 8. Build the trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

## 9. Train

This is the long-running cell. On a T4 expect 2-4 hours; on an A100 ~45 minutes.

In [ ]:
print("Starting training...")
trainer.train()

print("\nSaving final adapter...")
trainer.save_model(OUTPUT_DIR)               # saves LoRA adapter + tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

# Final evaluation
print("\nFinal eval...")
metrics = trainer.evaluate()
trainer.log_metrics("final_eval", metrics)
trainer.save_metrics("final_eval", metrics)
print(metrics)

if HF_REPO_ID:
    print(f"\nPushing adapter to Hub: {HF_REPO_ID}")
    trainer.push_to_hub()

print(f"\nDone. Adapter saved to: {OUTPUT_DIR}")

## 10. Free memory before inference

Training loaded the model in training mode with gradients; before inference we free everything and reload clean.

In [ ]:
import gc

del trainer
del model
gc.collect()
torch.cuda.empty_cache()
print("Memory freed.")

## 11. Inference — load the trained adapter

Loads the base Qwen2.5-3B-Instruct in 4-bit, then attaches the trained LoRA adapter on top.

In [ ]:
from peft import PeftModel

ADAPTER_PATH = OUTPUT_DIR

inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
inference_tokenizer.padding_side = "left"   # left-pad for generation

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)

# Attach the trained LoRA adapter
inference_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
inference_model.eval()
print("Adapter loaded. Ready for inference.")

## 12. Test the model on sample legal questions

In [ ]:
def ask(question: str, contract_excerpt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Contract excerpt:\n{contract_excerpt}\n\nQuestion: {question}"},
    ]
    prompt = inference_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

    with torch.no_grad():
        output = inference_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,            # deterministic for legal QA
            temperature=0.0,
            repetition_penalty=1.05,
            pad_token_id=inference_tokenizer.pad_token_id or inference_tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return inference_tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_contract = """
This Master Services Agreement ("Agreement") is entered into as of January 15, 2024,
between Acme Corp. ("Client") and Globex LLC ("Provider"). The initial term of this
Agreement shall be three (3) years from the Effective Date, and shall automatically
renew for successive one (1) year terms unless either party provides written notice
of non-renewal at least ninety (90) days prior to the end of the then-current term.
Either party may terminate this Agreement for material breach upon thirty (30) days
written notice, provided the breaching party has failed to cure such breach within
the notice period.
"""

questions = [
    "What is the initial term of this agreement?",
    "What is the notice period required for non-renewal?",
    "Under what conditions can a party terminate for material breach?",
    "What is the governing law of this contract?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q, sample_contract)}")

## Next steps

Once this adapter trains well on CUAD, the natural next milestones in your LexForge pipeline are:

1. **LegalBench evaluation harness** — builds the ROUGE-L regression gate that will live inside CI/CD Pipeline 3 (model eval).
2. **DPO alignment stage** — Direct Preference Optimization on the ~10k synthetic preference pairs for legal reasoning quality.
3. **Jurisdiction-specific adapters** — train separate LoRA adapters on EU (ECtHR) and India legal corpora, then serve them via vLLM's LoRA hot-swapping per request.
4. **Push adapter to HuggingFace Hub** — set `HF_REPO_ID` at the top of the notebook and re-run the training cell, or call `trainer.push_to_hub()` manually.